# Team Member Contribution Table

| Member      | Contribution |
| ----------- | ----------- |
| Petter      | Worked on SW|
| Kim (Scrum Master)        | Worked on ML |
| Elliot      | Worked on SW |
| Agust       | Research for next sprint |

## Model Development and Performance Improvement for AimoScore Regression adn WeakestLink Classification

This section presents the systematic improvement of the two models: The regression and Classification model. The objective for this sprint was to improve the performance of both models compared to its previous versions.

The development process included restructuring preprocessing pipelines, introducing cross-validation strategies, applying class balancing, experimenting with advanced ensemble methods, and evaluating alternative modeling architectures such as hierarchical and expert-based classifiers. The final modesl were selected based on objective evaluation metrics consistent with those used in the original notebooks.

## 2. Regression Model

### Baseline Model

The original regression model relied primarily on linear regression applied to a single train-test split. Evaluation metrics included:
* Coefficient determination (R^2)
* Root Mean Squared Error (RMSE)
* Mean Absolute Error (MAE)

The baseline linear regression achived:

`Test R^2 ~ 0.62`
`RMSE ~ 0.15`
`MAE ~0.11`

### Improvements

#### Gradient Boosted Regression

To address nonlinear relationships, gradient boosting models were introduced, specifically `LightGBM` and `XGBoost` (the winning models for weakest link). The final tuned `XGBoost` model achived:

`Test R^2 ~ 0.77`
`RMSE ~ 0.116`
`MAE ~ 0.083`

This represents a substantial improvement over the original baseline R^2 of approx. `0.62`

#### Cross-validation strategy for regression

A strucutred 5-fold-cross-validation framework was introduced during hyperparameter tuning. Rather than relying solely on a single train-test split, model selection was guided by cross-validated negative RMSE.

This change improved model reliability and reduced variance in performance estimates. Although tuning slightly adjusted performance, the dominant improvement stemmed from replacing linear regression with gradient boosting.

## 3. Classification Model

The original weakest_link_classification notebook evaluated several classical classifiers:
* Logisit regression
* Linear Discriminant Analysis
* k-Nearest Neighbors
* Gaussian Naive bayes
* XGBoost

Evaluated metrics included:
* Accuracy
* F1 (macro)
* F1 (weighted)

The weighted F1 score of the best earlier configuration was approx. 0.62, thought macro-F1 performance was lower due to class imbalance.

### Improvements

#### Stratified cross-validation
And adaptive StratifiedKFold strategy was introduced. The number of folds was dynamically determined based on the smallest class fequency to ensure that each fold contained representative samples. This improved stability in macro-level metrics and avoided folds with missing minority classes.

#### Boosted optimization
LightGBM and XGBoost were configured with increased capacity and tuned hyperparameters, including:
* Increased number of estimators
* Subsampling for regularization
* Feature subsampling
Depth and leaf optimization

## Results

Cross-validation results indicated the LightGBM performed best in terms of macro-F1 and weighted F1:

(Dropping classes < 2, not dropping ties)
`CV macro-F1 ~ 0.55`
`Test macro-F1 ~ 0.506`
`Test weighted F1 ~ 0.638`
`Test accuracy ~ 0.64`

and when we also drop ties we get:
`CV macro-F1 ~ 0.51`
`Test macro-F1 ~ 0.57`
`Test weighted F1 ~ 0.657`
`Test accuracy ~ 0.66`

The reason CV macro-F1 and weighted F1 moves in opposite directions is because weighted F1 accounts for class support and therefore assigns greater influence to majority classes, whereas macro F1 gives equal weight to each class irrespective of frequency. By removing tie cases, many ambiguous observations were eliminated, which generally improved predicted consistency for the dominan classes. Since these classes constitute a large proportion of the dataset, improvements in their precision  substantially increased the weighted F1-score. Hovewer, the removal of tied samples also reduced the number of training examples available for minority classes. In some cases, this led to reduced stability and weaker generalization performance for those smaller classes during cross-validation. Because macro F1 penalizes in minority class perfromance outweighed gains in larger classes, resulting in a lower macro F1-score.

## 4. Additional A4 experiments: Hierarchical and Expert Models

Several experimental architecutres were evaluated in the A4 research notebooks, including:
* `v2.ipynb`
* `v3(main).ipynb`
* `v3(expertmodels).ipynb`
* `v3(upperOrlowerOrtrunk).ipynb`
* `v4.ipynb`

### Hierarchical 

Some notebooks implemented a hierarchical classification strategy, diving the problem into anatomical categories such as upper body, lower body, and trunk. Separate models were trained for each subgroup.

The theoretical motivation was that specialized models might better deiscriminate among anatomically similar deviations.

However, empirical evaluation revealed that this decomposition reduced performance. The partitioning reduced effective training data per model, increaseing varaince and reducing generalization capacity. Additionally, misclassification in higher-level grouping stages propagated errors downstream.

All scores did not improve and, in most configurations, slightly declined relative to the unified boosting model.

#### Expert models

The `v3(expertmodels)` notebook tested training multiple specialized classifiers and combining predictions. While cinceptually appealing, this approach introduced additional complexity without measurable improvement in scores. 

The primary limitation was insuffiecient data per specialist model, leading to overfitting.

### Iterative model refinement (`v2.ipynb` and `v4.ipynb`)

These notebooks systematically adjusted hyperparameters and preprocessing techniques. Incremental improvements were observed, but none surpassed the final unified LightGBM configuration trained on the full dataset (80/20)

## Cross-validation approach and impact

The introduction fo strucutred cross-validation was a critical methodological improvement across both regression and classification tasks.

For regression, 5-fold cross-validation guided hyperparameter tuning and improved reliability of performance estimates.

For classification, stratified cross-validation ensured balanced representation of minority classes across folds. This reduced instability in macro-F1 and prevented misleading performance estimates driven by dominant classes.

Balanced weighting further improved macro-level metrics by correcting class imbalance bias.

## Conclusion
The regression model was substantially improved from an R^2 of approx 0.62 using linear regression to approx 0.77 using a tuned XGBoost regressor with target transformation. 

The classification model achived improved macro-F1 stability and competetive weighted F1 performance through balanced weighting, strucutred stratified cross-validation, and boosted ensemble tuning. Although hierarchical multi-model architectures were explored, they did not outperform the unifed LightGBM model.

# Improvements of client/server app and the DevOps/QA process

* Major improvements to the client/server app and the DevOps/QA process
* QA rules and how you enforce them

## General improvements
Several improvements have been made to the FastAPI application, including the integration of unit testing using pytest, linting using flake8, and a reorganization of the FastAPI codebase (see `ml/api`). In addition, linting has been introduced for the web application using ESLint and htmlhint for JavaScript and HTML respectively (see `sw`).

Unit testing was intentionally not implemented for the web application at this stage. The application is relatively small and simple, and manual testing is currently sufficient to verify its functionality. 

The FastAPI application was reorganized to incorporate design patterns that promote low coupling and improved modularity. This makes the code more maintainable and easier to test. These changes enhance the maintainability and testability of the code. For example, dependency injection is used in the routers (see `ml/api/app/routers`) to inject an instance of the model registry, which manages all registered models (see `ml/api/app/core/model_registry.py`). This approach decouples the routers from specific model implementations and allows the registry instance to be easily replaced during unit testing with a registry containing dummy models.

Below is a summary of the FastAPI files, including brief descriptions of their contents, their roles within the application, and the design patterns applied. 
```bash
.
├── app
│   ├── config.py
│   ├── core
│   │   ├── dummy_models.py
│   │   ├── model_loader.py
│   │   ├── model_registry.py
│   │   ├── model_type.py
│   │   ├── model.py
│   │   └── registry_instance.py
│   ├── dependencies
│   │   └── dependencies.py
│   ├── models.py
│   ├── routers
│   │   ├── features.py
│   │   ├── health.py
│   │   └── predict.py
│   └── services
│       └── inference.py
├── Dockerfile
├── main.py
├── README.txt
├── requirements-dev.txt
└── requirements.txt
```

* app/config.py - Defines basic configuration using Pydantic. This file also specifies paths to model files and input feature definitions.
* app/core/dummy_models.py - Contains placeholder model implementations used when no real model is available. Very useful for unit testing.
* app/core/model_loader.py - Defines a class with static methods responsible for loading model files and their corresponding input feature definitions. This encapsulates the model loading logic and separates it from other parts of the application.
* app/core/model_registry.py - Implements a model registry class responsible for storing and managing models along with their input features. The registry maintains a dictionary of models that can be retrieved when needed. The constructor accepts two callable parameters for loading models and features, respectively. This use of dependency injection improves maintainability and allows alternative implementations (such as mock loaders for unit testing) to be used. 
* app/core/model_type.py - Defines an enumeration class that specifies standardized identifiers for different models. For example, the weakest link model is identified by the name `weakest_link_model`. This ensures consistency and reduces the risk of errors caused by hardcoded strings.
* app/core/model.py - Defines a class that encapsulates a model instance together with its corresponding feature definitions. This ensures that models and their required input features are managed as a single logical unit.
* app/core/registry_instance.py - Initializes a shared model registry instance that can be imported and used throughout the application. For example, main.py uses this instance to register models before the API service starts.
* app/dependencies/dependencies.py - Provides dependency functions used by FastAPI. In particular, it defines a function that returns the shared registry instance from `app/core/registry_instance.py`. This function is used for dependency injection in routers, improving modularity and simplifying unit testing.
* app/models.py - Defines Pydantic models representing prediction request and response schemas. These ensure structured validation of incoming requests and consistent formatting of API responses.
* app/routers/features.py - Defines API endpoints for retrieving feature information. The FastAPI Depends mechanism is used to inject the model registry, ensuring loose coupling between components.
* app/routers/health.py - Defines API endpoints used to check the health and availability of the server. Right now, a user can only check if the server is active.
* app/routers/predict.py - Defines API endpoints for generating predictions. The model registry is injected using FastAPI's dependency injection system, allowing the router to access models without directly managing their initialization.
* app/services/inference.py - Contains the inference service logic. It defines a predict method that uses a provided model to generate predictions and returns a structured prediction response. This service layer separates business logic from the API layer, improving maintainability and testability.
* main.py - Serves as the application entry point. It initializes the FastAPI application, registers required models in the registry, and starts the API server.


The web application was not revised during this sprint since it is too small and simple. Incorporating any design patterns or architectures would not provide any significant value for now. 

## Quality assurance
We established a set of coding standards and guidelines to ensure consistency and maintainability across the codebase. These standards cover aspects such as proper commenting practices, naming conventions for variables and functions, code formatting rules (enforced using tools such as Flake8), testing practices, and the vertical organization of related code to improve readability (vertical density). The guidelines also provide recommendations on when and how to apply design patterns to promote modularity and low coupling.

Code formatting and certain quality aspects are enforced automatically through the use of linters and unit tests. These automated checks are integrated into our CI/CD pipeline, ensuring that code must meet the defined standards before it can be merged. Peer reviews are also used to evaluate higher-level quality aspects that cannot be fully enforced by automated tools.

Testing and code formatting are partially checked automatically using unit tests and linters. These checks are also included in our CI/CD to enforce these standards. We also use peer review to check for higher level formating rules that linting tools don't cover such as readability and vertical ordering, and if there are any important test cases that we are missing.

## Sprint 5 Preparation Report

To get a head start on Sprint 5, We analysed the upcoming requirements and focused on understanding them as a whole for the sprint. We have written issues for the sprint based on the given requriments from moodle. This will hopefully reduce startup time in the next sprint, since this sprint didnt have much work that could be split among us and we deduced that it would require more work to split the work than just doing the actual work. 

## For next sprint 
Notes for ourselves

Look at Lasso with polynomial features for regression model

Measure the quality of the models instead of just pushing the scores. Perform statistical comparison. Important: `T-test`. Stability of models, box plots. 

Notes:
Document is important. For example for a startup, when evaluating the "value" one would look at how easy it is to "onboard" oneself. This is important to appease investors etc in this example. 

